# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Selected Lane:** **Lane 2 — Refresh / Content Opportunity Scoring**

**Why this lane:**
FlyRank publishes content as infrastructure across client websites, but over time, high-performing articles experience organic search decay—impressions plateau, search rankings slip, and organic traffic quietly erodes. Human editorial teams possess strictly bounded bandwidth and cannot manually audit tens of thousands of existing URLs every month. 
A naive heuristic or arbitrary date-based refresh schedule either wastes editor hours updating zero-demand pages or completely misses high-visibility articles suffering urgent decay. By framing refresh identification as a machine-learned ranking and scoring problem over observable Google Search Console (GSC) and Google Analytics (GA4) signals, we can reliably surface the highest-leverage pages for human review first, outperforming static hand-tuned business rules.

In [1]:
# Lane definition and scope
LANE_NAME = "Refresh / Content Opportunity Scoring (Lane 2)"
PRIMARY_OBJECTIVE = "Prioritize existing pages for editorial refresh review based on decay risk and traffic opportunity"

print(f"Selected Lane: {LANE_NAME}")
print(f"Objective: {PRIMARY_OBJECTIVE}")

Selected Lane: Refresh / Content Opportunity Scoring (Lane 2)
Objective: Prioritize existing pages for editorial refresh review based on decay risk and traffic opportunity


## 2. The question: decision, action, cost of a wrong call

**1. What decision does this work improve?**
It answers the operational question: *"Given thousands of published URLs across diverse client domains, which 20 to 50 specific pages should an editorial team review and refresh first this week?"*

**2. Who acts on the output, and what do they do?**
The consumer is a **content strategist / editorial team**. When presented with a prioritized review queue, an editor inspects the page alongside transparent reason codes (e.g., decaying impression trajectory, high impressions with collapsing CTR, or outdated content age) and executes targeted remediation: refreshing factual content, improving metadata and snippet CTR, restructuring headings, or optimizing internal linking.

**3. What does a wrong recommendation cost?**
- **False Positive (recommending a healthy or zero-value page):** Wastes expensive human editorial capacity rewriting content that did not need updates or has no audience demand.
- **False Negative (missing a steeply decaying high-visibility page):** Leads to compounding organic traffic loss that becomes exponentially harder to recover once Google demotes the page's ranking authority.
- **Asymmetric Cost Structure:** Because editor capacity is strictly constrained, high **Precision@K (e.g., Precision@50)** is dramatically more critical than raw global recall across thousands of irrelevant low-traffic pages.

In [2]:
# Operational decision parameters
TARGET_DECISION = "Which pages to review and refresh first"
ACTOR = "Editorial team / Content Strategist"
TOP_K_REVIEW_CAPACITY = 50

print(f"Decision: {TARGET_DECISION}")
print(f"Actor: {ACTOR}")
print(f"Evaluation Horizon: Top-{TOP_K_REVIEW_CAPACITY} queue precision")

Decision: Which pages to review and refresh first
Actor: Editorial team / Content Strategist
Evaluation Horizon: Top-50 queue precision


## 3. Quick look at the data (2-3 real numbers)

To verify empirical viability, we inspect `data/raw/content_refresh_anonymized.csv` (30,000 pseudonymized content items across 32 clients) and compute core operational metrics.

In [3]:
import os
import pandas as pd
import numpy as np

# Resolve path relative to notebook location or repo root
data_path = "../../data/raw/content_refresh_anonymized.csv" if os.path.exists("../../data/raw/content_refresh_anonymized.csv") else "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

total_pages = len(df)
num_clients = df["client_id"].nunique()

# 1. Prevalence of organic search decay
declining_mask = df["trend_direction"].str.lower().eq("down")
n_declining = declining_mask.sum()
pct_declining = (n_declining / total_pages) * 100

# 2. Substantial traffic exposure at risk (pages with >= 500 impressions in trailing 90 days)
visible_mask = df["impressions_90d"] >= 500
n_visible = visible_mask.sum()
pct_visible = (n_visible / total_pages) * 100

# 3. High-demand pages in striking distance (position 1-20) suffering sub-par CTR (< 0.5%)
striking_low_ctr = ((df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & 
                    (df["avg_position"] <= 20) & (df["ctr"] < 0.5)).sum()
pct_striking_low_ctr = (striking_low_ctr / total_pages) * 100

print(f"Dataset Scope: {total_pages:,} pages across {num_clients} pseudonymized clients.")
print(f"Real Number 1 (Decay Prevalence): {n_declining:,} pages ({pct_declining:.1f}%) exhibit negative impression momentum ('down').")
print(f"Real Number 2 (High Exposure Pool): {n_visible:,} pages ({pct_visible:.1f}%) have substantial visibility (>= 500 impressions/90d).")
print(f"Real Number 3 (Striking Distance Opportunity): {striking_low_ctr:,} pages ({pct_striking_low_ctr:.1f}%) occupy page 1-2 search results but capture < 0.5% CTR.")

Dataset Scope: 30,000 pages across 32 pseudonymized clients.
Real Number 1 (Decay Prevalence): 16,262 pages (54.2%) exhibit negative impression momentum ('down').
Real Number 2 (High Exposure Pool): 16,726 pages (55.8%) have substantial visibility (>= 500 impressions/90d).
Real Number 3 (Striking Distance Opportunity): 9,759 pages (32.5%) occupy page 1-2 search results but capture < 0.5% CTR.


## 4. Careful words: what I can and can't claim

Following the FlyRank research guidelines and data safety ethics, we establish strict disciplinary boundaries on our scientific claims:

### What we CAN claim:
- **Observed / Measured relationships:** We can document measured associations between pre-decision features (such as `days_with_impressions`, `avg_position`, historical CTR, and engagement rates) and page momentum trajectories.
- **Directional prioritization:** We can produce an evidence-backed ranking that assigns higher probability of decay and refresh urgency to specific pages.
- **Decision-support utility:** We can prove through rigorous client-holdout validation that a learned ranking model achieves superior precision (e.g., Precision@50 ~ 0.74 vs 0.24) compared to fixed heuristic rules.

### What we CANNOT and will NEVER claim:
- **No causal guarantees:** We cannot claim that executing an editorial refresh is guaranteed to cause ranking recovery. Proving causality requires an active randomized A/B trial or quasi-experimental difference-in-differences design with counterfactual controls.
- **No reverse-engineering of Google:** We do not claim to have "cracked or predicted Google's ranking algorithm." Google's core ranking systems encompass thousands of live signals, personalization layers, and external backlink graphs not present in this dataset.
- **No private domain exposure:** No raw client URLs, client domains, or unhashed queries are revealed; all analysis relies on pseudonymized IDs and aggregate statistical distributions.

In [4]:
# Discipline claim checklist
allowed_claims = [
    "Observed historical traffic and engagement patterns",
    "Directional prioritization of pages exhibiting decay risk",
    "Decision-support ranking to optimize manual editorial review bandwidth",
    "Measured lift in Precision@K over static rule baselines under client-holdout splits"
]

disallowed_claims = [
    "Causal proof that refreshing a page guarantees search rank recovery",
    "Reverse-engineering Google's proprietary search ranking algorithm",
    "Automated production publishing without human-in-the-loop editorial verification"
]

print("ALLOWED CLAIMS:")
for c in allowed_claims:
    print(f"  [+] {c}")

print("\nDISALLOWED CLAIMS:")
for c in disallowed_claims:
    print(f"  [-] {c}")

ALLOWED CLAIMS:
  [+] Observed historical traffic and engagement patterns
  [+] Directional prioritization of pages exhibiting decay risk
  [+] Decision-support ranking to optimize manual editorial review bandwidth
  [+] Measured lift in Precision@K over static rule baselines under client-holdout splits

DISALLOWED CLAIMS:
  [-] Causal proof that refreshing a page guarantees search rank recovery
  [-] Reverse-engineering Google's proprietary search ranking algorithm
  [-] Automated production publishing without human-in-the-loop editorial verification


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.